# 01 — Data exploration (EDA)

**Capstone:** Smart Portfolio system  
**Purpose:** Load adjusted closes from Yahoo Finance, inspect prices and **log returns**, and visualize **correlation** across assets.

This mirrors the data path used in `backend/optimize.py` (`fetch_price_data`, `compute_returns`).


In [ ]:
# Allow imports from backend/ (start Jupyter from `notebooks/` or project root)
from pathlib import Path
import sys
for base in [Path.cwd(), Path.cwd() / "notebooks"]:
    for b in [base, base.parent]:
        be = b / "backend"
        if (be / "optimize.py").exists():
            sys.path.insert(0, str(be))
            break
    else:
        continue
    break
else:
    raise RuntimeError("Could not find backend/ — `cd` to smart-portfolio-system or notebooks/")


In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
START, END = "2022-01-01", "2024-12-31"

from optimize import fetch_price_data, compute_returns

prices = fetch_price_data(TICKERS, START, END)
prices.head()


## Price levels (normalized to 1 at start for comparison)


In [ ]:
import matplotlib.pyplot as plt

norm = prices / prices.iloc[0]
ax = norm.plot(figsize=(10, 5), title="Normalized prices")
ax.set_ylabel("Index (start = 1)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Log returns — distribution and correlation


In [ ]:
rets = compute_returns(prices)
rets.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
rets.plot(ax=axes[0], alpha=0.7, title="Daily log returns")
axes[0].set_ylabel("log return")

import seaborn as sns
sns.heatmap(rets.corr(), annot=True, fmt=".2f", cmap="vlag", ax=axes[1])
axes[1].set_title("Correlation of daily log returns")
plt.tight_layout()
plt.show()


### Takeaway
- Missing rows are dropped in `fetch_price_data` (`dropna`).
- The engine uses **sample** mean and covariance of log returns (annualized in `estimate_parameters`). Strong correlation implies limited diversification in-sample.
